# Fine-tune EDM CIFAR-10 → CIFAR-100

Fine-tunes NVIDIA's pretrained CIFAR-10 EDM diffusion model on CIFAR-100, following the [MimicDiffusion](https://github.com/psky1111/MimicDiffusion) recipe: fine-tune the **unconditional** EDM checkpoint. This avoids the class-count mismatch you'd get fine-tuning the conditional checkpoint (10-way label embedding vs. CIFAR-100's 100 classes).

**Before running:** `Runtime` → `Change runtime type` → select a GPU (T4 or better).

**Persistence:** set `USE_DRIVE = True` in the config cell and mount Drive to keep the prepared dataset zip, downloaded checkpoint, and training runs/snapshots on Drive so nothing is lost if the Colab session terminates -- no need to "Save a copy in Drive" of this notebook itself, since it's tracked on GitHub. Three things are deliberately kept off Drive even with `USE_DRIVE=True`: the cloned edm repo itself (we `%cd` into it -- a Drive hiccup mid-training would take the shell's own working directory down with it, not just a file read); the raw 50k-PNG CIFAR-100 scratch dump (disposable, and Drive's per-file overhead makes writing that many small files there very slow); and the copy of the dataset zip actually passed to training (DataLoader workers read it continuously throughout training, and that read cache resets to cold on every resumed run -- reading it live from Drive re-exposes ~50k individual Drive reads right after every reconnect). The canonical dataset zip still lives on Drive; training just reads from a local copy of it. If training gets interrupted mid-run, just re-run the fine-tuning cell -- it auto-detects the latest checkpoint under `outdir` on Drive and resumes from there instead of starting over.

Source of truth for this workflow: [`finetune_edm_cifar100.py`](./finetune_edm_cifar100.py) in this repo. If you hit an issue running this notebook, report it back in the Claude Code session that generated it -- fixes land in that `.py` file (and this notebook) and get pushed to this branch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import glob
import os
import shutil
import torchvision

# ── Configuration ──
USE_DRIVE = True  # read/write everything under Google Drive
drive_base_path = '/content/drive/MyDrive/FineTunedCheckpoint/edm-cifar100'  # only used if USE_DRIVE
# Paths below are quoted before being passed to shell magics, so spaces are
# tolerated -- but note this path is unrelated to where the notebook file
# itself lives; it's just where this script writes its own working files.

BASE = drive_base_path if USE_DRIVE else '/content'
# NOTE: BASE is not created here -- see the next cell. Creating
# '/content/drive/...' locally before Drive is mounted there makes
# drive.mount() refuse to mount ("Mountpoint must not already contain
# files"), since it then finds a non-empty local directory sitting at the
# mount point instead of an empty one.

COND = False  # unconditional fine-tuning (recommended -- see markdown cell above).
# COND=True is possible but the label-embedding weights won't transfer
# (shape mismatch: 10 classes -> 100), so --transfer effectively
# reinitializes that layer at random; only the backbone benefits.

DURATION_MIMG = 10  # fine-tuning budget, in millions of images
                    # (the paper's from-scratch CIFAR-10 run used 200; fine-tuning needs far less)
BATCH = 128         # effective/logical batch size (affects training dynamics, e.g. the
                    # EMA halflife schedule) -- lower than the paper's default of 512, but
                    # this alone does NOT bound GPU memory; scale --tick/--snap if you
                    # raise DURATION_MIMG a lot
BATCH_GPU = 32      # actual per-step minibatch size; train.py accumulates gradients over
                    # BATCH // BATCH_GPU steps to reach BATCH, so this is what actually
                    # controls peak GPU memory. With no --batch-gpu, a single-GPU run
                    # processes all of BATCH in one forward/backward pass, which OOMs on
                    # a ~15GB GPU (T4) at BATCH=128 for this architecture; drop this
                    # further (16, 8, ...) if you still hit OOM

TICK_KIMG = 10   # images (in thousands) per tick -- just a progress/logging unit
SNAP_TICKS = 2   # write a network-snapshot-*.pkl every SNAP_TICKS * TICK_KIMG images
                 # (currently 20k)
DUMP_TICKS = 2   # write a training-state-*.pt (what --resume needs) every
                 # DUMP_TICKS * TICK_KIMG images (currently 20k). This is your
                 # worst-case progress loss on a disconnect -- tightened from
                 # the (10, 10) defaults (100k-image loss window) after losing
                 # a real run's progress to a disconnect that landed before the
                 # first checkpoint ever got written. Lower further (e.g. 1) if
                 # disconnects keep happening well inside this window; each
                 # dump costs some Drive I/O time and space, so don't go
                 # extreme without reason

In [ ]:
# Mount Drive (only if USE_DRIVE=True above), THEN create BASE.
# Order matters: creating BASE before this would create a local stub at
# /content/drive that blocks the mount (see note in the cell above).
#
# os.path.ismount() only checks /content/drive is a distinct mount point --
# it stays True even if the underlying Drive FUSE connection has gone stale
# (e.g. after a network hiccup or long idle period), which then fails every
# file access with "[Errno 107] Transport endpoint is not connected".
# Actually probe it with a real read and force a remount if that fails.
if USE_DRIVE:
    from google.colab import drive
    needs_mount = True
    if os.path.ismount('/content/drive'):
        try:
            os.listdir('/content/drive/MyDrive')
            needs_mount = False
            print("Google Drive is already mounted and responsive at /content/drive")
        except OSError:
            print("Google Drive mount is stale (not responding) -- remounting...")
    if needs_mount:
        drive.mount('/content/drive', force_remount=True)

os.makedirs(BASE, exist_ok=True)

In [ ]:
# 1. Clone the NVlabs/edm repository.
# Deliberately local, not under BASE, even with USE_DRIVE=True: this is
# where we `%cd`, so if it lived on Drive, any Drive hiccup during training
# (mount drop, network blip) would take down the shell's own cwd along with
# it -- "getcwd: cannot access parent directories: Transport endpoint is
# not connected" -- not just a file read. The clone itself is a few
# seconds' work and gains nothing from persistence; only the
# dataset/checkpoint/training-run *data* below actually needs Drive.
edm_dir = '/content/edm'
if not os.path.isdir(edm_dir):
    !git clone https://github.com/NVlabs/edm.git "{edm_dir}"

%cd "{edm_dir}"

# EDM (2022) predates several current-PyTorch/single-GPU-Colab realities.
# Patch the affected lines in the cloned repo; str.replace is a no-op on
# repeat runs, so this is safe to run every time regardless of whether the
# repo was just cloned or already existed.
def _patch_file(rel_path, old, new, description):
    path = os.path.join(edm_dir, rel_path)
    with open(path) as f:
        src = f.read()
    if old not in src:
        print(f"WARNING: expected text not found in {rel_path}, skipping patch: {description}")
        return
    patched = src.replace(old, new)
    if patched != src:
        with open(path, 'w') as f:
            f.write(patched)
        print(f"Patched {rel_path}: {description}")

# 1) torch.utils.data.Sampler's __init__ used to take a `data_source` arg;
# modern PyTorch's no longer does, so InfiniteSampler's
# `super().__init__(dataset)` raises "TypeError: object.__init__() takes
# exactly one argument".
_patch_file(
    'torch_utils/misc.py',
    'super().__init__(dataset)', 'super().__init__()',
    "InfiniteSampler for modern PyTorch's Sampler.__init__()",
)

# 2) dist.init() hardcodes backend='nccl' on any non-Windows OS, even
# though we only ever run a single process (no torchrun/multi-GPU) where
# gloo works identically. This avoids "RuntimeError: Distributed package
# doesn't have NCCL built in" on any torch build without NCCL compiled in
# (e.g. a CPU-only build, if the runtime doesn't actually have a GPU
# attached -- check with !nvidia-smi if you hit this).
_patch_file(
    'torch_utils/distributed.py',
    "backend = 'gloo' if os.name == 'nt' else 'nccl'",
    "backend = 'gloo' if os.name == 'nt' or not torch.distributed.is_nccl_available() else 'nccl'",
    "fall back to gloo backend when NCCL isn't compiled in",
)

# 3) PyTorch 2.6 changed torch.load()'s default from weights_only=False to
# weights_only=True. --resume's training-state-*.pt contains pickled
# custom objects (e.g. torch_utils.persistence._reconstruct_persistent_obj)
# that trip the new default's unpickling restriction, raising
# "Weights only load failed ... WeightsUnpickler error: Unsupported
# global". This is EDM's own file (written by this same script's earlier
# run), not an untrusted download, so weights_only=False is safe here.
_patch_file(
    'training/training_loop.py',
    "data = torch.load(resume_state_dump, map_location=torch.device('cpu'))",
    "data = torch.load(resume_state_dump, map_location=torch.device('cpu'), weights_only=False)",
    "load training-state checkpoints with weights_only=False for PyTorch 2.6+",
)

# 4) train.py's Logger duplicates *every single print()* to a log.txt under
# run_dir, flushing on every write since should_flush=True. Since run_dir
# is under outdir (on Drive when USE_DRIVE=True), this means any Drive
# hiccup -- at any point during the whole run, not just at checkpoint
# writes -- crashes the process immediately with
# "OSError: [Errno 107] Transport endpoint is not connected", as seen
# crashing on the very first print (the network summary table) before any
# training even started. Point the log file at local disk instead; it's
# just a human-readable log, not something that needs Drive persistence
# (checkpoints/snapshots are still written to Drive separately, on their
# own controlled schedule).
_patch_file(
    'train.py',
    "os.path.join(c.run_dir, 'log.txt')",
    "f'/content/edm-log-{os.path.basename(c.run_dir)}.txt'",
    "write the stdout log locally instead of through the Drive mount",
)

In [ ]:
# 2. Install dependencies.
# The repo ships environment.yml (a conda spec), not a pip requirements.txt
# -- `pip install -r environment.yml` fails trying to parse YAML as
# requirements. Colab already has a CUDA-matched torch/numpy/pillow/scipy,
# so we deliberately don't force environment.yml's pinned torch==1.12.1
# (that would fight the preinstalled CUDA build for no benefit); just
# pip-install the packages listed there.
!pip install "numpy>=1.20" "click>=8.0" "pillow>=8.3.1" "scipy>=1.7.1" psutil requests tqdm imageio "imageio-ffmpeg>=0.4.3" pyspng

In [ ]:
# 3. Prepare the CIFAR-100 dataset.
# dataset_tool.py only derives labels from a dataset.json or from top-level
# subfolder names -- NOT from filenames -- so save each image into a
# per-class subfolder (train/<label>/*.png) rather than one flat folder.
#
# The raw PNG dump is kept on local disk (not under BASE) even when
# USE_DRIVE=True: it's disposable scratch input to dataset_tool.py below,
# and writing ~50k individual files through Drive's FUSE mount is slow.
local_scratch = '/content/cifar100-scratch'
raw_dir = os.path.join(local_scratch, 'train')
os.makedirs(raw_dir, exist_ok=True)

dataset_zip = os.path.join(BASE, 'datasets', 'cifar100-32x32.zip')
os.makedirs(os.path.dirname(dataset_zip), exist_ok=True)

if os.path.isfile(dataset_zip):
    # Already prepared and persisted (e.g. from a previous session on Drive)
    print(f"Found existing prepared dataset at {dataset_zip}, skipping re-prep.")
else:
    trainset = torchvision.datasets.CIFAR100(
        root=os.path.join(local_scratch, 'data_temp'), train=True, download=True,
    )
    for i, (img, label) in enumerate(trainset):
        class_dir = os.path.join(raw_dir, f'{label:03d}')
        os.makedirs(class_dir, exist_ok=True)
        img.save(os.path.join(class_dir, f'{i:05d}.png'))
    print(f"CIFAR-100 training images saved to {raw_dir}")

    print(f"Converting to EDM dataset format ({dataset_zip})...")
    !python dataset_tool.py --source="{raw_dir}" --dest="{dataset_zip}" --resolution=32x32

# Copy to local disk for actual training use: DataLoader workers read
# --data continuously throughout training, and that read cache resets to
# cold on every resumed run (fresh process). Reading it live from Drive
# re-exposes ~50k individual Drive reads right after every reconnect --
# exactly when we've seen crashes cluster. dataset_zip on Drive remains the
# persisted, canonical copy; train.py reads from this local copy instead.
local_dataset_zip = '/content/cifar100-32x32.zip'
if not os.path.isfile(local_dataset_zip):
    print(f"Copying dataset to local disk for training ({local_dataset_zip})...")
    shutil.copy(dataset_zip, local_dataset_zip)

In [ ]:
# 4. Download the pretrained EDM CIFAR-10 checkpoint (NVIDIA ships .pkl, not .pt)
checkpoints_dir = os.path.join(BASE, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)
ckpt_name = 'edm-cifar10-32x32-cond-vp.pkl' if COND else 'edm-cifar10-32x32-uncond-vp.pkl'
ckpt_path = os.path.join(checkpoints_dir, ckpt_name)
!wget -nc https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/{ckpt_name} -P "{checkpoints_dir}"

In [ ]:
# 5. Fine-tune.
outdir = os.path.join(BASE, 'training-runs-cifar100')
os.makedirs(outdir, exist_ok=True)

# If a previous run under outdir got interrupted (Colab disconnect, runtime
# recycle, etc.), --dump periodically wrote a full training-state file
# (optimizer + step count, not just weights) there. Pick the latest one and
# --resume from it instead of --transfer-ing the pretrained checkpoint again
# -- that's what actually avoids losing progress on session termination.
resume_candidates = sorted(glob.glob(os.path.join(outdir, '*', 'training-state-*.pt')))
resume_path = resume_candidates[-1] if resume_candidates else None

if resume_path:
    print(f"Found existing training state, resuming from {resume_path}")
    weight_arg = f'--resume="{resume_path}"'
else:
    print(f"No existing training state found, transferring pretrained weights from {ckpt_path}")
    weight_arg = f'--transfer="{ckpt_path}"'

print(f"batch={BATCH}, batch_gpu={BATCH_GPU}")

# --duration is in millions of images, not kimg. --data points at the local
# copy of the dataset zip (see step 3), not the Drive-persisted one. There's
# no --metrics flag on this train.py -- FID/metric computation lives in the
# separate fid.py script, run manually against a snapshot after training.
train_cmd = (
    f'python train.py '
    f'--outdir="{outdir}" '
    f'--data="{local_dataset_zip}" '
    f'--cond={"1" if COND else "0"} '
    f'{weight_arg} '
    f'--duration={DURATION_MIMG} '
    f'--batch={BATCH} '
    f'--batch-gpu={BATCH_GPU} '
    f'--tick={TICK_KIMG} '
    f'--snap={SNAP_TICKS} '
    f'--dump={DUMP_TICKS}'
)
print(train_cmd)
!{train_cmd}

# Don't just assume success -- check train.py's actual exit code (IPython
# sets the special _exit_code var after every `!` command) and confirm a
# snapshot really landed on Drive, rather than printing "finished"
# unconditionally regardless of what happened.
snapshots = sorted(glob.glob(os.path.join(outdir, '*', 'network-snapshot-*.pkl')))
if _exit_code != 0:
    print(f"\n*** train.py exited with code {_exit_code} -- it did NOT complete successfully. ***")
    if snapshots:
        print(f"Latest snapshot checkpointed on Drive before the failure: {snapshots[-1]}")
        print("Re-run this cell to resume training from there.")
    else:
        print("No snapshot was checkpointed yet -- re-running this cell will restart from the pretrained checkpoint.")
elif snapshots:
    print(f"\nTraining completed. Final network snapshot saved to Drive:\n  {snapshots[-1]}")
else:
    print("\ntrain.py exited cleanly but no network-snapshot-*.pkl was found under outdir -- this is unexpected, check the run's local log (/content/edm-log-*.txt).")